# The polynomial Stokes problem of Araya et al. (2017)

We use the exact velocity and pressure from §3.1.1 of [the 2017 paper](https://doi.org/10.1016/j.cma.2017.05.027): a streamfunction $\psi=-128x^2(x-1)^2y^2(y-1)^2$, $u=(\psi_y,-\psi_x)$, and $p=150(x-1/2)(y-1/2)$. Sources are differentiated from these fields. These diagonal triangulations and P2/P1 or P1/P1 local spaces differ from the paper's complete convergence experiments. Assembly uses Duffy order 5 and error integration uses order 8, sufficient for the respective polynomial degrees.

The application declares both the local and global equations; the portable
provider uses the existing basis, quadrature and sparse-scatter kernels. For
Taylor–Hood the local bilinear form is

$$
\begin{aligned}
a_T((u,p),(v,q))&=(\nabla u,\nabla v)_T
 -(p,\operatorname{div}v)_T-(q,\operatorname{div}u)_T,\\
B_T\lambda&=\langle\lambda,v\rangle_{\partial T},\qquad
C_T(u,p)=-\langle u,\mu\rangle_{\partial T}.
\end{aligned}
$$

The P1/P1 USFEM form subtracts the residual pairing with
$R(u,p)=\nabla p$ and
$\tau_T=h_T^2/24$ for unit viscosity and zero drag. Its RHS is
$(f,v)_T-\tau_T(f,\nabla q)_T$. The P1 velocity Laplacian vanishes
on each fine triangle, so the inverse constant is exactly $m=1/3$.
The global `Equation` explicitly supplies negative Dirichlet velocity moments;
the pressure gauge integrates the reconstructed physical pressure.


The PyMHM distribution contains only the library. The first cell explicitly
downloads a checksum-verified companion archive and prepares the declared data
using its local Python helpers. You can inspect these support files in `ROOT`.
Complete studies and historical replay retain their existing opt-in flags.


In [ ]:
from pathlib import Path
import os
import sys
from pymhm.io.workspace import workspace_from_archive

# Download verified support files; this operation does not execute them.
COMPANION_URL = "https://ipes-lncc.github.io/pymhm/downloads/a69357413fea92e5ab0fa3f38f5faa48900c1cb2a55335fca9854949dfb58f48/05_stokes_published_problem-companion.zip"
COMPANION_SHA256 = "a69357413fea92e5ab0fa3f38f5faa48900c1cb2a55335fca9854949dfb58f48"
WORKSPACE = Path(
    os.environ.get("PYMHM_WORKSPACE", Path.cwd() / ".pymhm-companions" / COMPANION_SHA256)
)
ROOT = workspace_from_archive(COMPANION_URL, sha256=COMPANION_SHA256, directory=WORKSPACE)
os.environ["PYMHM_WORKSPACE"] = str(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

# Explicitly prepare declared data with the downloaded Python helpers.
from scripts.notebook_reproduction import notebook_workspace

ROOT = notebook_workspace("flow/stokes/05_stokes_published_problem.ipynb", directory=ROOT)
root = ROOT
print("Workspace:", ROOT)


In [ ]:
from functools import partial
import numpy as np
from pymhm import Equation, MultiscaleProblem, assemble, TriangleMesh, SkeletonSpace, FaceSpace
from pymhm.fem.scalar.operators import boundary_data
from examples.formulations.flow import (
    VelocityPressureSpace, velocity_pressure_equations,
    velocity_pressure_fields, p1_residual_weight,
)


In [ ]:
from numpy.polynomial import Polynomial
a=Polynomial([0,0,1,-2,1])
def velocity(x):
    """Differentiate the streamfunction to obtain exactly solenoidal velocity."""
    X,Y=x.T
    return 128*np.column_stack((-a(X)*a.deriv()(Y),a.deriv()(X)*a(Y)))
def pressure(x):
    """Return the analytical zero-mean pressure."""
    return 150*(x[:,0]-.5)*(x[:,1]-.5)
def source(x):
    """Return negative velocity Laplacian plus the pressure gradient."""
    X,Y=x.T
    lap=128*np.column_stack((-(a.deriv(2)(X)*a.deriv()(Y)+a(X)*a.deriv(3)(Y)),a.deriv(3)(X)*a(Y)+a.deriv()(X)*a.deriv(2)(Y)))
    return -lap+150*np.column_stack((Y-.5,X-.5))
for formulation in ('taylor-hood','usfem'):
    errors=[]
    for n in (2,4):
        mesh=TriangleMesh.unit_square(n)
        space=SkeletonSpace(mesh,tuple(FaceSpace.uniform(1) for _ in mesh.faces),2)
        degree = 2 if formulation == "taylor-hood" else 1
        pressure_degree = degree - 1 if formulation == "taylor-hood" else degree
        local_space = VelocityPressureSpace(mesh, space, degree, pressure_degree, 4, 5)
        provider = partial(
            velocity_pressure_equations, space=local_space, source=source,
            residual_weight=p1_residual_weight if formulation == "usfem" else None,
        )
        boundary, _ = boundary_data(space, velocity, order=5)
        problem = MultiscaleProblem(
            Equation(0, -np.r_[boundary, np.zeros(2 * len(mesh.cells))]),
            provider, range(len(mesh.cells)), space.size, (2,) * len(mesh.cells),
        )
        system = assemble(problem)
        pressure_gauge = system.mean_constraint(
            [record[2] for record in system.local_metadata], 0.0,
        )
        coefficients = system.solve(constraints=(pressure_gauge,))
        result = velocity_pressure_fields(system, coefficients, local_space)
        errors.append(result.l2_error(velocity,order=8))
        print(formulation,n,'velocity',errors[-1],'pressure',result.pressure_l2_error(pressure,order=8),'divergence',result.divergence_l2())
    assert errors[1]<errors[0]/3


The discrete velocity is weakly incompressible and need not have pointwise zero divergence. A global pressure gauge is essential. Residual convergence and a divergence-free analytical velocity do not alone prove pointwise discrete incompressibility.